# Project: Instance-Based (Support Vector Machine (SVM))
## Name: Jesed Renteria


In [ ]:
#Example of supress warnings for Numpy version out of range (optional)
import warnings
warnings.filterwarnings("ignore", category=Warning)
warnings.simplefilter(action='ignore', category=FutureWarning)

#Some recommended libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import learning_curve

# The Dataset

In [ ]:
# Load the Parkinson's Disease dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/parkinsons.data"
df = pd.read_csv(url)
print(df.head())
print(df.info())
#df = pd.get_dummies(df, columns = ["name"])
df2 = df.drop(columns = ["name"])
#No null values
# Target variables is Status

In [ ]:
df["status"]

# Data Preprocessing

In [ ]:
corr_matrix = df2.corr()
plt.imshow(corr_matrix, cmap='coolwarm', interpolation='nearest')
plt.colorbar()
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.index)), corr_matrix.index)
plt.title('Correlation Matrix')
plt.show()
# Status seems to have a strong correlations with spread1, spread2, PPE, MDVP:Fo(Hz), and MDVP:Fhi(Hz) 

In [ ]:
#Standardization # Train-Test Splitting
scaler = StandardScaler()
X = df.drop(columns=['name', 'status'])  # Features
y = df['status']  # Target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# Building the SVM Model

In [ ]:
#Choosing hyperparameters
# The kernel type specifies what kind of kernel to use in the algorithm
# The regularization parameter, C, is used to control the trade-off between maximizing the margin and minimizing misclassification
# High C, greatly minimizes misclassifications but creates a smaller margin which overfits the data
# Low C, greatly maximizes the margin but allows some misclassifications, potentially underfitting the model.
# gamma define how far the influence of a training data point extends to affect the decision boundary.
svm = SVC(kernel='rbf', C=1, gamma='scale')
svm.fit(X_train, y_train)

#Make predictions
y_pred = svm.predict(X_test)


# Evaluating the Model

In [ ]:
#Insert Code Here
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average = "weighted")
recall = recall_score(y_test, y_pred, average = "weighted")
f1 = f1_score(y_test, y_pred, average = "weighted")

conf_matrix = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(conf_matrix, "\n")
print(f"Accuracy: {accuracy}", '\n')
print(f"Precision: {precision}", '\n')
print(f"Recall: {recall}", '\n')
print(f"F1 Score: {f1}", '\n')

## For the Model Selection Project, you will STOP HERE! 
During Units 4, 5, and 6, we will explore and learn additional techniques, and then revisit these projects to apply the below:
- Model evaluation and parameter tuning
- Explanatory visualizations and package your results with data storytelling

# Tuning Model Parameters (Completed in Unit 4)

In [ ]:
#Insert Code Here
param_grid1 = {'C': [0.1, 1, 10, 100],
              'gamma': [1, 0.1, 0.01, 0.001], 
              'kernel': ['linear', 'rbf', 'poly']}
grid_search = GridSearchCV(estimator=SVC(), param_grid=param_grid1, cv=5, scoring='accuracy', verbose=2, n_jobs=-1)
grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)


# Evaluating the Tuned Model (Completed in Unit 4)

In [ ]:
#Insert Code Here
print("Best hyperparameters: ", grid_search.best_params_)
print("Best score (negative RMSE): ", grid_search.best_score_)


accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(conf_matrix, "\n")
print(f"Accuracy: {accuracy}", '\n')
print(f"Precision: {precision}", '\n')
print(f"Recall: {recall}", '\n')
print(f"F1 Score: {f1}", '\n')

# Visualizing Results (Completed in Units 4 and 6)

In [ ]:

# Compute confusion matrix
c_matrix = confusion_matrix(y_test, y_pred)

class_names = ['Healthy', 'Parkinson’s'] 

# Plot using seaborn
plt.figure(figsize=(6, 5))
sns.heatmap(c_matrix, annot=True, fmt='d', cmap='Greens', 
            xticklabels=class_names, yticklabels=class_names)

plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
#Insert Code Here
plt.figure(figsize=(10, 6))
train_sizes, train_scores, test_scores = learning_curve(
    estimator = best_model,
    X = X_train,
    y = y_train,
    train_sizes = np.linspace(0.1, 1.0, 5),
    cv = 5,
    scoring = "accuracy",
    n_jobs = -1
)
# Mean and Standard Deviations for Training and Testing subsets
train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_mean, 'o-', color='blue', label='Training Accuracy')
plt.plot(train_sizes, test_mean, 'o-', color='green', label='Validation Accuracy')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
plt.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color='green')

plt.title('Learning Curve for SVC (Best Parameters)')
plt.xlabel('Number of Training Samples')
plt.ylabel('Accuracy')
plt.legend(loc='best')
plt.grid(True)
plt.tight_layout()
plt.show()


    Audience: 
    This presentation will aimed toward data scientists, people with Parkinson's disease, researchers and healthcare providers. The stakeholders would be the researchers and healthcare providers that deal have a deeper understanding of Parkinson's. They may know the criteria and diagnosis of Parkinson's, not to mention they might be able to have ideas on how to prevent the development of Parkinson's once they figure out that their patient has Parkinson's. As for data scientists, they would be interested in the model that is being used to predict Parkinson's disease to either improve. As for people with Parkinson's, they will feel inspired from our presentation to do something and spread our work towards people with Parkinson's or to the community.
    
    Introduction: 
    The dataset that will be used is the Parkinson's Disease dataset which contains various biomedical voice measurements from people with and without Parkinson's and the problem that is being tackled is the prediction of Parkinson's disease.
    
    Data Preprocessing:
    During the data preprocessing stage, the dataset was loaded into a dataset using the Pandas library in python. Then, the dataset was checked to see if there were any null values using the ".info()" function call that displayed the information of the dataset such as features of the dataset, non-null count and the datatype. Seeing as how the only feature with words is the name feature, it was removed to keep people's personal information private and it did not seem necessary to keep it to build the model. After everything was cleaned up, visuals were made specifically the correlation matrix to provide an understanding of the target variable, status, that predicts Parkinson's, using the matplotlib library and the ".corr()" function call that returns a dataframe representing the correlation matrix, and noticed that four features had pretty high correlation with the target variable, two of which had a negative association and the other two had a positive association. Then, the target variable, status, is separated from the dataset into a variable, and the rest of features into another variable. Then, the rest of the features is standardized, using the "StandardScaler()" function call from the Scikit-Learn library, to have a mean of 0 and a standard deviation of 1 for consistent training and to ensure that the model is not biased towards features with larger scales.
    
    Model Architecture: 
    The model that was selected is the Support Vector Machine which is used for classification and regression tasks containing hyperparameters such as kernel type, regularization paramter, and gamma using the Scikit-learn library to build the model. The kernel type parameter is used to specify what kind of kernel to use in the algorithm, the gamma parameter defines how far the influence of a training data point extends to affect the decision boundary, and the regularization parameter, C, is used to control the trade-off between maximizing the margin and minimizing misclassification. High C, greatly minimizes misclassifications but creates a smaller margin which overfits the data. Low C, greatly maximizes the margin but allows some misclassifications, potentially underfitting the model. The parameter that was selected for the model's kernel type is "rbf" (Radial Basis Function), which has the ability to map data to higher-dimensional spaces, making non-linear patterns easier to find. As for the regularization parameter, it was set to equal 1, and as for gamma, it was set to "scale" to set the gamma based on the data's scale, directly impacting how to model fits the decision boundary.

    
    Training and Evaluation: 
    The training involved splitting the dataset into training and testing sets to evaluate the model's performance on unseen data. 80% of the dataset goes to the training subset and 20% of it goes to the testing subsets, using the "train_test_split()" function call from the Scikit-Learn library. After building the model to predict Parkinson's Disease and getting the needed metrics to evaluate the model, the model had an accuracy of 89.7%, a precision of 90.88%, a recall of 89.7% and a F1 score of 87.99%.
    

    Parameter Tuning:
    To find the optimal hyperparameters and their outcomes, a grid search, from the Scikit-Learn library, was used to experiment different combinations of the the Support Vector Machine to find the best estimator. As a result, the best estimator had the regularization parameter of 100, gamma of 0.1, and kernel type of "rbf". When comparing the metrics for the best estimator, it was noticed the metrics improved with an accuracy of 94.8%, a precision of 94.1%, a recall of 100%, and an F1 score of 96.9%.
    
    Conclusion: 
    Since working with the Parkinson's Disease dataset, it helped myself understand how to use Support Vector Machines and how to tune it to improve the model to predict Parkinson's. Potential improvements for this project would be to research further for more parameters of the Support Vector Machine or other techniques to improve the model.
